# ElevenLabs · Lesson 02: Advanced TTS — Streaming, Stitching & APIs

Beyond basic TTS, ElevenLabs supports **streaming** for real-time playback,
**request stitching** for natural multi-paragraph audio, and serves perfectly as a
**FastAPI streaming endpoint**.

## What you will learn
1. **In-memory streaming** — `BytesIO` for chunk-based audio
2. **Request stitching** — `previous_request_ids` for prosody continuity
3. **Multi-paragraph narration** — stitching a whole story together
4. **Output format tradeoffs** — quality vs file size vs latency
5. **Building a TTS API** — FastAPI endpoint walkthrough

---
> **Prerequisites:** Lesson 01 completed. ElevenLabs SDK installed and `.env` configured.

In [1]:
# === Setup ===
import sys
sys.path.insert(0, '.')

from _helpers import setup_client, play_audio, save_audio, timer

client = setup_client()

ElevenLabs client ready (key: ...10a3e9)


---
## 1. In-Memory Streaming

Instead of saving to a file, we can collect audio chunks in a `BytesIO` buffer.
This is useful for:
- Processing audio without touching disk
- Forwarding audio to another service
- Building real-time applications

In [2]:
# Streaming σε μνήμη — BytesIO buffer
from io import BytesIO
from elevenlabs import VoiceSettings

def text_to_speech_stream(text, voice_id="JBFqnCBsd6RMkjVDRZzb"):
    """Generate TTS and return as in-memory BytesIO stream."""
    response = client.text_to_speech.convert(
        voice_id=voice_id,
        output_format="mp3_22050_32",
        text=text,
        model_id="eleven_turbo_v2_5",
        voice_settings=VoiceSettings(
            stability=0.5,
            similarity_boost=1.0,
            style=0.0,
            use_speaker_boost=False,
            speed=1.0
        )
    )

    audio_stream = BytesIO()
    chunk_count = 0
    for chunk in response:
        if chunk:
            audio_stream.write(chunk)
            chunk_count += 1

    audio_stream.seek(0)
    return audio_stream, chunk_count


with timer("Stream generation"):
    stream, chunks = text_to_speech_stream(
        "Technology has transformed education. From interactive whiteboards to tablets, "
        "new avenues for learning are opening every day."
    )

print(f"📦 Received {chunks} chunks, total size: {len(stream.getvalue()) / 1024:.1f} KB")
play_audio(stream.getvalue())

⏱️ Stream generation: 1399 ms
📦 Received 33 chunks, total size: 32.6 KB


---
## 2. Request Stitching — The Key Technique

When you generate multiple audio segments separately, each one starts with a "fresh" prosody.
This creates **jarring transitions** between segments.

**Request stitching** solves this by passing the IDs of previous requests,
so the model continues with consistent intonation and rhythm.

### How it works:
1. Use `with_raw_response.convert()` instead of `convert()`
2. Extract the `request-id` header from each response
3. Pass the **last 3** request IDs in `previous_request_ids`

```
Paragraph 1  →  request-id: "abc123"     →  previous_request_ids: []
Paragraph 2  →  request-id: "def456"     →  previous_request_ids: ["abc123"]
Paragraph 3  →  request-id: "ghi789"     →  previous_request_ids: ["abc123", "def456"]
Paragraph 4  →  request-id: "jkl012"     →  previous_request_ids: ["abc123", "def456", "ghi789"]
Paragraph 5  →  request-id: ...           →  previous_request_ids: ["def456", "ghi789", "jkl012"]  (sliding window!)
```

In [3]:
# Request Stitching — Αλυσίδα αιτημάτων για φυσικό ήχο
from io import BytesIO
import time

paragraphs = [
    "The advent of technology has transformed countless sectors, with education "
    "standing out as one of the most significantly impacted fields.",
    
    "In recent years, educational technology, or EdTech, has revolutionized the way "
    "teachers deliver instruction and students absorb information.",
    
    "From interactive whiteboards to individual tablets loaded with educational software, "
    "technology has opened up new avenues for learning that were previously unimaginable.",
    
    "One of the primary benefits of technology in education is the accessibility it provides.",
]

# --- Stitched version ---
request_ids = []
audio_buffers = []

start = time.perf_counter()
for i, paragraph in enumerate(paragraphs):
    with client.text_to_speech.with_raw_response.convert(
        text=paragraph,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        previous_request_ids=request_ids[-3:]   # ← Sliding window of last 3!
    ) as response:
        # Αποθήκευση request-id από τα headers
        req_id = response._response.headers.get("request-id")
        request_ids.append(req_id)
        
        audio_data = b''.join(chunk for chunk in response.data)
        audio_buffers.append(BytesIO(audio_data))
        
        print(f"  Paragraph {i+1}: request-id = {req_id[:16]}... | size = {len(audio_data)/1024:.1f} KB")

stitched_ms = (time.perf_counter() - start) * 1000

# Συνδυασμός όλων των buffers
combined = BytesIO(b''.join(buf.getvalue() for buf in audio_buffers))

print(f"\n✅ Stitched {len(paragraphs)} paragraphs in {stitched_ms:.0f} ms")
print(f"📦 Total size: {len(combined.getvalue()) / 1024:.1f} KB")

save_audio(combined.getvalue(), "lesson02_stitched.mp3")
play_audio(combined.getvalue())

  Paragraph 1: request-id = Y5t5ycCz2MMbGHNr... | size = 122.9 KB
  Paragraph 2: request-id = 18j2kHv2HLkpFuQB... | size = 133.1 KB
  Paragraph 3: request-id = Mutrj8HUfFc1YiZq... | size = 155.6 KB
  Paragraph 4: request-id = SK4muS4Y8ALHWy2a... | size = 80.0 KB

✅ Stitched 4 paragraphs in 6250 ms
📦 Total size: 491.6 KB
💾 Saved: outputs\lesson02_stitched.mp3 (491.6 KB)


> 🎧 **Listen carefully** to the transitions between paragraphs.
> The intonation should flow naturally, as if one continuous reading.

---
## 3. Without Stitching (for comparison)

Let's generate the same paragraphs **without** stitching to hear the difference:

In [4]:
# Χωρίς stitching — κάθε paragraph ανεξάρτητα

unstitched_buffers = []

start = time.perf_counter()
for paragraph in paragraphs:
    audio = client.text_to_speech.convert(
        text=paragraph,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        # Χωρίς previous_request_ids!
    )
    audio_bytes = b''.join(chunk for chunk in audio if chunk)
    unstitched_buffers.append(audio_bytes)

unstitched_ms = (time.perf_counter() - start) * 1000

combined_unstitched = b''.join(unstitched_buffers)

print(f"⏱️ Unstitched: {unstitched_ms:.0f} ms")
print(f"⏱️ Stitched:   {stitched_ms:.0f} ms (from above)")

save_audio(combined_unstitched, "lesson02_unstitched.mp3")
print("\n🎧 Unstitched version:")
play_audio(combined_unstitched)

⏱️ Unstitched: 5992 ms
⏱️ Stitched:   6250 ms (from above)
💾 Saved: outputs\lesson02_unstitched.mp3 (505.5 KB)

🎧 Unstitched version:


> 🔍 **Compare:** Open both `lesson02_stitched.mp3` and `lesson02_unstitched.mp3`.
> Notice how the stitched version flows naturally while the unstitched version
> has "restarts" at each paragraph boundary.

---
## 4. Output Format Comparison

| Format | Sample Rate | Bitrate | Quality | Size | Use Case |
|--------|-------------|---------|---------|------|----------|
| `mp3_44100_128` | 44.1 kHz | 128 kbps | ⭐⭐⭐⭐⭐ | Large | Production, podcasts |
| `mp3_22050_32` | 22 kHz | 32 kbps | ⭐⭐⭐ | Small | Mobile, previews |
| `pcm_16000` | 16 kHz | Raw PCM | ⭐⭐⭐⭐ | Very large | Processing pipelines |
| `ulaw_8000` | 8 kHz | μ-law | ⭐⭐ | Tiny | Telephony |

In [ ]:
# Σύγκριση μεγέθους αρχείου ανά format

test_text = "This sentence will be generated in multiple audio formats to compare quality and size."

formats = [
    ("mp3_44100_128", "MP3 High (44.1kHz/128kbps)"),
    ("mp3_22050_32",  "MP3 Low (22kHz/32kbps)"),
]

print(f"{'Format':<35} {'Size':>10} {'Ratio':>8}")
print("-" * 55)

sizes = []
for fmt, label in formats:
    audio = client.text_to_speech.convert(
        text=test_text,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        output_format=fmt
    )
    audio_bytes = b"".join(chunk for chunk in audio if chunk)
    size_kb = len(audio_bytes) / 1024
    sizes.append(size_kb)
    ratio = f"{size_kb / sizes[0]:.2f}x" if sizes[0] > 0 else "1.00x"
    print(f"{label:<35} {size_kb:>8.1f} KB {ratio:>8}")

---
## 5. Building a TTS API with FastAPI

You can serve TTS as a web API using FastAPI with `StreamingResponse`.
This code can't run in a notebook, but here's the complete, working implementation:

```python
# tts_fastapi.py — Run with: uvicorn tts_fastapi:app
import os
from dotenv import load_dotenv
from elevenlabs import VoiceSettings
from elevenlabs.client import ElevenLabs
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

load_dotenv()
app = FastAPI()
client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))


def tts_stream(text: str):
    """Generator that yields audio chunks."""
    response = client.text_to_speech.convert(
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        output_format="mp3_22050_32",
        text=text,
        model_id="eleven_turbo_v2_5",
        voice_settings=VoiceSettings(
            stability=0.5, similarity_boost=1.0, speed=1.0
        )
    )
    for chunk in response:
        if chunk:
            yield chunk


@app.get("/tts")
async def tts_endpoint(text: str):
    return StreamingResponse(
        tts_stream(text),
        media_type="audio/mpeg"
    )
```

**Usage:**
```bash
pip install fastapi uvicorn
uvicorn tts_fastapi:app --reload
# Then open: http://localhost:8000/tts?text=Hello+World
```

> 💡 The browser will directly play the audio from the URL!

> ✏️ **Exercise:** Build a request-stitched audiobook from a 5-paragraph text.
> 1. Split any article into 5 paragraphs
> 2. Generate each with request stitching
> 3. Combine and save as a single MP3
> 4. Measure total latency and compare with/without stitching

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **`BytesIO` streaming** | Collect audio chunks in memory without touching disk |
| **Request stitching** | Pass `previous_request_ids[-3:]` for natural prosody across segments |
| **`with_raw_response`** | Access HTTP headers (including `request-id`) from API responses |
| **Sliding window** | Only the last 3 request IDs are used — older ones don't help |
| **Output formats** | `mp3_44100_128` for quality, `mp3_22050_32` for speed, `pcm` for processing |
| **FastAPI + StreamingResponse** | Serve TTS as a web API with `media_type="audio/mpeg"` |

---
**Next lesson:** Pronunciation Dictionaries — teaching the AI how to say names correctly